## はじめに

このノートブックでは、GlacierStyle ECサイトの各種テーブルにメタデータ情報を自動付与します。

**主な処理内容:**
- AI_GENERATE_TABLE_DESCによるテーブル・カラム説明の自動生成
- TRANSLATEによる日本語翻訳
- セマンティックビューの作成（Cortex Analyst向け）

In [1]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [2]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part3/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [3]:
display_image('architecture.png')

## 3. 各種テーブルにメタデータ情報を付与

AI_GENERATE_TABLE_DESCとTRANSLATEを組み合わせて、テーブル・カラムに日本語コメントを自動付与します。

### 3-1. Description自動生成

ストアドプロシージャを使用して、テーブルとカラムの説明を自動生成します。

**処理フロー:**
1. AI_GENERATE_TABLE_DESC: テーブルデータを分析し、英語で説明を自動生成
2. SNOWFLAKE.CORTEX.TRANSLATE: 生成された説明を日本語に翻訳
3. ALTER TABLE/ALTER COLUMN: コメントとして設定

**対象テーブル:**
- ディメンジョンテーブル: dim_customers, dim_products
- ファクトテーブル: fact_orders, fact_payments, fact_web_logs
- Gold層テーブル: gold_sns_mentions_analyzed, gold_voice_logs, gold_ad_creative_analysis, gold_faq_documents, gold_operation_manuals, gold_sns_mentions_with_product_master

In [4]:
display_image('ai_generate_description.png')

In [14]:
-- ============================================================================
-- SPROC作成: テーブル・カラムコメント自動生成
-- ============================================================================
-- AI_GENERATE_TABLE_DESCでテーブル情報を分析し、TRANSLATEで日本語に翻訳
-- 生成された説明をテーブル・カラムのコメントとして設定
CREATE OR REPLACE PROCEDURE set_table_comments_ja(
    table_name_param STRING
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
import json

def main(session, table_name_param):
    try:
        sql_generate = f"""
            CALL AI_GENERATE_TABLE_DESC(
                '{table_name_param}',
                OBJECT_CONSTRUCT('describe_columns', true, 'use_table_data', true)
            )
        """
        result = session.sql(sql_generate).collect()
        output = json.loads(result[0][0])
        
        table_info = output['TABLE'][0]
        table_desc_en = table_info['description']
        table_desc_en_escaped = table_desc_en.replace("'", "''")
        
        sql_translate_table = f"""
            SELECT SNOWFLAKE.CORTEX.TRANSLATE('{table_desc_en_escaped}', 'en', 'ja')
        """
        table_desc_ja_result = session.sql(sql_translate_table).collect()
        table_desc_ja = table_desc_ja_result[0][0]
        table_desc_ja_escaped = table_desc_ja.replace("'", "''")
        
        sql_set_table_comment = f"ALTER TABLE {table_name_param} SET COMMENT = '{table_desc_ja_escaped}'"
        session.sql(sql_set_table_comment).collect()
        
        columns = output.get('COLUMNS', [])
        for column in columns:
            column_name = column['name']
            column_desc_en = column['description']
            column_desc_en_escaped = column_desc_en.replace("'", "''")
            
            sql_translate_col = f"""
                SELECT SNOWFLAKE.CORTEX.TRANSLATE('{column_desc_en_escaped}', 'en', 'ja')
            """
            column_desc_ja_result = session.sql(sql_translate_col).collect()
            column_desc_ja = column_desc_ja_result[0][0]
            column_desc_ja_escaped = column_desc_ja.replace("'", "''")
            
            sql_set_col_comment = f"ALTER TABLE {table_name_param} ALTER COLUMN {column_name} COMMENT '{column_desc_ja_escaped}'"
            session.sql(sql_set_col_comment).collect()
        
        return f'SUCCESS: {table_name_param}'
    except Exception as e:
        return f'ERROR: {str(e)}'
$$;

In [ ]:
-- ============================================================================
-- 各テーブルにメタデータコメントを付与
-- ============================================================================
CALL set_table_comments_ja('dim_customers');

In [17]:
-- 休憩の前に実行する 
-- 4分くらいかかる模様
-- ============================================================================
-- 各テーブルにメタデータコメントを付与 
-- ============================================================================
-- ディメンジョンテーブル
CALL set_table_comments_ja('dim_products');

-- ファクトテーブル
CALL set_table_comments_ja('fact_orders');
CALL set_table_comments_ja('fact_payments');
CALL set_table_comments_ja('fact_web_logs');

-- Gold層テーブル
CALL set_table_comments_ja('gold_sns_mentions_analyzed');
CALL set_table_comments_ja('gold_voice_logs');
CALL set_table_comments_ja('gold_ad_creative_analysis');
CALL set_table_comments_ja('gold_faq_documents');
CALL set_table_comments_ja('gold_operation_manuals');
CALL set_table_comments_ja('gold_sns_mentions_with_product_master');

In [16]:
-- ============================================================================
-- メタデータ付与結果の確認
-- ============================================================================
-- テーブル・カラムに設定されたコメントを確認
DESC TABLE dim_customers;

### 3-2. セマンティックビュー作成

Cortex Analyst向けのセマンティックビューを作成します。

**セマンティックビューとは:**
- ビジネスメトリクスやエンティティの関係を定義
- 自然言語クエリ（Text-to-SQL）を可能にする
- ビジネス用語の同義語（シノニム）を定義

**作成方法:**
- オプション1: Snowsight UI上でウィザードを使用
- オプション2: SQLクエリで直接作成

### オプション1. Snowsight上での作成

1. 次のいずれかの方法で、セマンティックビューを作成するためのウィザードにアクセスします。

`1-1. データベースオブジェクトエクスプローラー`:
- Snowsight にサインインします。
- ナビゲーションメニューで Catalog » Database Explorer を選択します。
- セマンティックビューを作成するデータベースとスキーマを選択します。
- Create » Semantic View » Create with guided setup を選択します。

`1-2. Cortex Analyst`:
- Snowsight にサインインします。
- ナビゲーションメニューで AI & ML » Cortex Analyst を選択します。
- Create new » Create new Semantic View を選択します。


2. ウィザードの Getting started ステップで次を実行します。
- Location to store から、モデルを格納するデータベースとスキーマを選択します。
- Name に、セマンティックビューの名前を入力します。
  - 文字またはアンダースコアで始まり、文字、数字、アンダースコア、ドル記号のみを含む名前を指定する必要があります。
- （オプション） Description で、セマンティックビューによって利用可能になるデータの説明をします。
- Next を選択します。


3. `ウィザードの Select tables ステップで`:
- All タブで、セマンティックビューで使用するデータを含むテーブルまたはビューを選択します。次の点に注意してください。
  - 少なくとも1つのテーブルまたは表示を選択する必要があります。
  - パフォーマンスを向上させるために、10以上のテーブルを選択しないでください。
  - 選択したテーブルとビューのリストを表示するには、 Selected タブを選択します。
- Next を選択します。


4. `ウィザードの Select columns ステップで:`
- ビューに含める列を選択します。
  - テーブルまたは表示内のすべての列を選択するには、テーブルまたはビューを選択します。
  - パフォーマンス向上のため、50列以上は選択しないでください。
- Create and Save を選択します。


5. `Logical tables 中`:
- 各テーブルまたはビューに定義されているファクト、ディメンジョン、およびメトリクスを確認します。
- ビジネスに適した名称と説明を提供します。
- 必要なファクト、ディメンジョン、メトリクスを追加します。

6. `Relationships 中`:
- ジェネレーターによって定義された関係を確認します。
- 必要に応じてリレーションシップのプロパティを変更します。
- 必要なリレーションシップを追加してください。
- セマンティックビューに変更を加えた場合は、Save を選択します。

7. セマンティックビューに変更を加えた場合は、Save を選択します。

In [5]:
display_image('semantic_view_copilot.png')

### オプション2. SQLクエリでの作成

In [6]:
display_image('create_semantic_view.png')

In [18]:
-- GlacierStyle EC分析用セマンティックビューの作成
CREATE OR REPLACE SEMANTIC VIEW glacierstyle_db.ec_analytics_schema.ec_analysis_semantic_view
    TABLES (
        customers AS glacierstyle_db.ec_analytics_schema.dim_customers PRIMARY KEY (customer_id),
        products AS glacierstyle_db.ec_analytics_schema.dim_products PRIMARY KEY (product_id),
        orders AS glacierstyle_db.ec_analytics_schema.fact_orders PRIMARY KEY (order_id),
        payments AS glacierstyle_db.ec_analytics_schema.fact_payments PRIMARY KEY (payment_id),
        web_logs AS glacierstyle_db.ec_analytics_schema.fact_web_logs PRIMARY KEY (log_id)
    )
    RELATIONSHIPS (
        orders_to_customers AS orders (customer_id) REFERENCES customers,
        orders_to_products AS orders (product_id) REFERENCES products,
        payments_to_orders AS payments (order_id) REFERENCES orders
    )
    FACTS (
        orders.order_total AS total_amount,
        payments.pay_amt AS payment_amount
    )
    DIMENSIONS (
        customers.cust_name AS CONCAT(last_name, first_name),
        products.prod_name AS product_name,
        orders.order_date AS order_datetime
    )
    METRICS (
        orders.total_sales AS SUM(total_amount),
        orders.order_count AS COUNT(order_id)
    )
    COMMENT = 'GlacierStyle ECサイト分析用セマンティックビュー';

In [19]:
-- ============================================================================
-- セマンティックビューのクエリ発行
-- ============================================================================
SELECT 
    order_date,
    AGG(total_sales)
FROM glacierstyle_db.ec_analytics_schema.ec_analysis_semantic_view
GROUP BY order_date
ORDER BY order_date
LIMIT 10


In [20]:
-- ============================================================================
-- セマンティックビューの確認
-- ============================================================================
-- 作成したセマンティックビューの定義を確認
DESCRIBE SEMANTIC VIEW ec_analysis_semantic_view;

## まとめ

このノートブックでは、GlacierStyle ECサイトの各種テーブルにメタデータ情報を自動付与しました。

### 実施した処理

**3-1. テーブル・カラムコメントの自動生成**
- AI_GENERATE_TABLE_DESCでテーブル構造とデータを分析し説明を生成
- TRANSLATEで日本語に翻訳
- ALTER TABLE/ALTER COLUMNでコメントとして設定

**3-2. セマンティックビューの作成**
- Cortex Analyst向けのec_analysis_semantic_viewを作成
- TABLES: 5つのベーステーブル（customers, products, orders, payments, web_logs）
- RELATIONSHIPS: テーブル間の関係を定義
- FACTS/DIMENSIONS/METRICS: ビジネス指標と分析軸を定義
- SYNONYMS: 日本語の同義語を設定

### 対象テーブル一覧

**Dimension層**
- dim_customers: 顧客マスタ
- dim_products: 商品マスタ

**Fact層**
- fact_orders: 注文トランザクション
- fact_payments: 決済トランザクション
- fact_web_logs: Webアクセスログ

**Gold層**
- gold_sns_mentions_analyzed: SNS分析結果
- gold_voice_logs: 音声ログ分析結果
- gold_ad_creative_analysis: 広告クリエイティブ分析結果
- gold_faq_documents: FAQドキュメント
- gold_operation_manuals: 運用マニュアル
- gold_sns_mentions_with_product_master: SNS×商品マスタ突合結果

### 使用したCortex AI関数

- `AI_GENERATE_TABLE_DESC`: テーブル・カラム説明の自動生成
- `TRANSLATE`: 多言語翻訳（英語→日本語）

### 次のステップ

- Cortex Analystを使用した自然言語クエリの実行
- 各種AIエージェントの構築